In [ ]:
#With importlib.reload(module) I can update my utils and do not restart the kernel every time
import importlib
# import my_utils.detector_v4 as ud
# import my_utils.data_analysis_v4 as uda
# import my_utils.plt_style as ps

import h5py
import hdf5plugin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import cv2

from tqdm import tqdm   #progressbar

from scipy.optimize import curve_fit
import matplotlib.ticker as mticker
from matplotlib.animation import FuncAnimation
from IPython.display import display, Image as IPImage

# ps.article_style()

## Looking for t0 at sample

In [ ]:
day = '' 
scan_name = ''

f_name = f'{day}/{scan_name}'
print(f"Percorso: {f_name}")

all_entries = next(os.walk(f_name))[2]
im_names = sorted([f for f in all_entries if f.endswith('.h5') and not f.startswith('.')])
# Cartella dove salvare i dataset
ds_folder = os.path.join(f_name, 'ds')

if not os.path.exists(ds_folder):
    os.makedirs(ds_folder)
    print(f"✅ Cartella creata: {ds_folder}")
else:
    print(f"📁 La cartella esiste già: {ds_folder}")

#folder to save plots
plot_folder = f'{day}/{scan_name}'
if not os.path.exists(plot_folder):
    os.makedirs(plot_folder)

# Crea i percorsi completi per i file
file_names = [os.path.join(f_name, im_n) for im_n in im_names]
ds_folder = os.path.join(f_name, 'ds')

print(f"File trovati: {len(file_names)}")
if len(file_names) > 0:
    print(f"Primo file valido: {file_names[0]}")

## Sanity check, ROI selection and theshold set

Definisce threshold per rimuovere spikes e rumore...

In [ ]:
def hdf5_to_array(fileName, roi_L = 512, roi_x0 = 0, roi_y0 = 0):
    '''
    Read the hdf5 file and return the image as a numpy array of
    
    Params:
    fileName: str, path to the hdf5 file
    roi_L: int, length of the region of interest (ROI) in pixels
    roi_x0: int, x coordinate of the top left corner of the ROI
    roi_y0: int, y coordinate of the top left corner of the ROI   
    Returns:
    im: numpy array, the image data in the ROI
    '''
    
        
    with h5py.File(fileName, 'r') as f:
        dset = f['entry']['data']['data']   #this is how the h5 file is structured
        
        im = np.array(dset[:, roi_y0:(roi_y0+roi_L), roi_x0:(roi_x0+roi_L)], dtype=np.int32)
        
    return im

In [ ]:
import plotly.express as px
import nbformat

check_im = hdf5_to_array(file_names[0]).sum(0)
counts_ref_frame = check_im.sum()


fig = px.imshow(check_im, color_continuous_scale='Viridis', title="Use to set center and roi")
fig.show()

In [ ]:
rough_xc, rough_yc = 296, 282  ##CHANGE ME
roi_L = 512
roi_x0, roi_y0 = rough_xc-roi_L//2, rough_yc-roi_L//2

#plot the image with the ROI and print tot sum count
#check different files and use this to set the threshold
check_frame = hdf5_to_array(fileName = file_names[-10], roi_L=roi_L, roi_x0=roi_x0, roi_y0=roi_y0)[0,:,:]
plt.imshow(check_frame)
plt.title('Image with ROI')
plt.colorbar()

check_frame_sum = check_frame.sum()
print('Sum counts in the ROI: ', check_frame_sum)

########CHECK ME##############
threshold = check_frame_sum * 10
print('Assigned threshold: ', threshold)

## Frame alignement  
Each file containes multiple images, that I call frames. These frames need to be aligned, summed and stored

In [ ]:
def align_images(image, reference_image, number_of_iterations = 1000, termination_eps = 1e-9):
    '''
    Aligns the input image to the reference image using the Enhanced Correlation Coefficient (ECC) algorithm.
    The ECC algorithm maximizes the correlation coefficient between the input image and the reference image
    by estimating a transformation matrix iteratively. This method is robust and widely used for image alignment tasks.
    
    The function uses a translation motion model to align the images. It initializes a 2x3 transformation matrix
    to the identity matrix and iteratively updates it to maximize the correlation coefficient. The aligned image
    is obtained by applying the inverse of the transformation matrix to the input image.
    
    Reference:
    Evangelidis, G. D., & Psarakis, E. Z. (2008). Parametric Image Alignment using Enhanced Correlation Coefficient (ECC) Maximization.
    IEEE Transactions on Pattern Analysis and Machine Intelligence, 30(10), 1858–1865. https://doi.org/10.1109/TPAMI.2008.113
    
    Params:
    image: np.ndarray, the input image to be aligned
    reference_image: np.ndarray, the reference image to align to
    number_of_iterations: int, the maximum number of iterations for the ECC algorithm (default: 10)
    termination_eps: float, the threshold for the increment in the correlation coefficient between iterations (default: 1e-3)
    
    Returns:
    aligned_image: np.ndarray, the aligned image
    '''
    
    # Define the motion model
    warp_mode = cv2.MOTION_TRANSLATION
    
    # Define 2x3 transformation matrix and initialize to identity
    warp_matrix = np.eye(2, 3, dtype=np.float32) #float necessary for algorithm
    
    
    # Define termination criteria
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, number_of_iterations, termination_eps)
    
    # Run the ECC algorithm to find the warp matrix
    cc, warp_matrix = cv2.findTransformECC(reference_image.astype(np.float32), image.astype(np.float32), warp_matrix, warp_mode, criteria)
    
    # Use warpAffine for Translation, Euclidean and Affine
    aligned_image = cv2.warpAffine(image.astype(np.float32), warp_matrix, 
                                   (reference_image.astype(np.float32).shape[1], reference_image.astype(np.float32).shape[0]),
                                   flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP)
    
    return aligned_image

### Test

In [ ]:
ref_frame = check_frame.copy()
counts_ref_frame = ref_frame.sum()

#you can change the number in [] to check different frames
to_be_aligned_frame = hdf5_to_array(file_names[-10], roi_L, roi_x0, roi_y0)[-1,:,:]
# # Normalize the image to the same number of counts as the reference frame
# to_be_aligned_frame = to_be_aligned_frame/to_be_aligned_frame.sum() * counts_ref_frame

test_aligned_frame = align_images(to_be_aligned_frame, ref_frame)


#####Comparison plot
fig, axs = plt.subplots(2,2, dpi=300)
vmax = 9  ##change me
val = 5 ##change me

im = axs[0,0].imshow(ref_frame, vmax = vmax)
axs[0,0].set_title('Reference frame')
#colorbar
cbar = plt.colorbar(im, ax=axs[0,0])

axs[0,1].imshow(to_be_aligned_frame, vmax = vmax)
axs[0,1].set_title('Frame to be aligned')


#difference between the two images
im2 = axs[1,0].imshow(ref_frame - to_be_aligned_frame, cmap='seismic', vmin=-val, vmax=val)
axs[1,0].set_title('Original difference')
#colorbar
cbar = plt.colorbar(im2, ax=axs[1,0])

axs[1,1].imshow(ref_frame - test_aligned_frame, cmap='seismic', vmin=-val, vmax=val)
axs[1,1].set_title('Aligned difference')

plt.tight_layout()

### Aligning and discarding outliars (no normalization!)

In [ ]:
def get_image(file_name, ref_frame):
    '''
    Reads the hdf5 file, aligns all frames that have a number of counts below the threshold and sums them to create the final image.
    The frames that are above the threshold are discarded.
    The function uses the Enhanced Correlation Coefficient (ECC) algorithm to align the images.
    The frames for which the alignment fails are discarded.
    The final image is normalized to account for the discarded frames.
    
    Params:
    file_name: str, path to the hdf5 file
    ref_frame: np.ndarray, the reference frame to align to
    Returns:
    im: np.ndarray, the aligned image data
    counts: list, the number of counts in each frame
    '''
    
    # Read the hdf5 file and get the image data
    frames = hdf5_to_array(file_name, roi_L, roi_x0, roi_y0)  # (N_frames, L, L)
    
    # The final image
    im = np.zeros_like(frames[0], dtype=np.float64)
    # Store the counts for sanity check
    counts = []
    # Align the frames to the reference frame
    for i, frame in enumerate(frames):
        if frame.sum() < threshold:
            try:
                # Align the image to the reference frame
                aligned_frame = align_images(frame, ref_frame)
                
                # Sum the aligned frames
                im += aligned_frame
                
                # Update counts
                counts.append(frame.sum())

            except cv2.error as e:
                print(f"ECC alignment failed: frame {i} discarded\nFile: {file_name}")
                # Plot the image
                plt.imshow(frame, vmax=val)
                plt.title('Bad frame')
                plt.show()
                
                # Update counts
                counts.append(0)
        else:
            print(f'Frame {i} above threshold discarded. Frame counts = {frame.sum()}\nFile: {file_name}')
            # Update counts
            counts.append(0)
    
    # Normalize the final image to account for discarded frames
    if len(counts) > 0:
        im *= len(frames) / len([c for c in counts if c > 0])
    
    return im, counts

In [ ]:
#check function
im, counts = get_image(file_names[0], ref_frame)

fig, axs = plt.subplots(1, 2)

# Plot the aligned image
axs[0].imshow(im, cmap='viridis')
axs[0].set_title('Aligned Image')
axs[0].axis('off')

# Plot the counts
axs[1].plot(counts, marker='o', linestyle='-')
axs[1].set_title('Sum of Aligned Frames')
axs[1].set_xlabel('Frame Index')
axs[1].set_ylabel('Counts')

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
from tqdm import tqdm

# 1. Definiamo i percorsi in modo corretto per macOS
data_path = os.path.join(ds_folder, 'aligned_images_no_norm.npy')
counts_path = os.path.join(ds_folder, 'counts.npy')

# 2. CONTROLLO: Se i file esistono già, li carichiamo e saltiamo il lavoro lungo
if os.path.exists(data_path) and os.path.exists(counts_path):
    print("🚀 File .npy trovati! Caricamento in corso (salto il loop)...")
    data = np.load(data_path)
    counts = np.load(counts_path)
    print(f"✅ Dati caricati correttamente. Shape: {data.shape}")
else:
    print("⏳ Dati non trovati. Inizio elaborazione immagini (richiederà tempo)...")
    data = []
    counts = []

    # loop over all files
    for fn in tqdm(file_names):
        # Assicurati che 'ref_frame' sia stato definito correttamente prima
        im, count = get_image(fn, ref_frame)
        data.append(im)
        counts.append(count)
        
    # Conversione in array
    data = np.array(data)
    counts = np.array(counts)

    # 3. SALVATAGGIO: Ora salviamo i file per la prossima volta
    np.save(data_path, data)
    np.save(counts_path, counts)
    print(f" Dati salvati con successo in: {ds_folder}")

In [ ]:
# open the data
data = np.load(os.path.join(ds_folder, 'aligned_images_no_norm.npy'))
counts_per_frame = np.load(os.path.join(ds_folder, 'counts.npy'))

# sanity check plot of counts per frame
fig = px.line(y=counts_per_frame.flatten(), title='Counts for all frames, sanity check')
fig.update_layout(xaxis_title='Frame Index', yaxis_title='Counts')
fig.show()

## Dataset creation

In [ ]:
def get_coordinates(im_names):
    '''
    Get the coordinates of the images from the file names.

    This function extracts metadata from the image file names, including lab time, fast scan index, delay time, and theta.
    The file names are expected to follow a specific format where the metadata is encoded as underscores-separated values.

    Params:
    im_names: list of str, the list of image file names

    Returns:
    lab_time: np.ndarray, array of datetime64[us] representing the lab time for each image
    fast_scans: np.ndarray, array of integers representing the fast scan index for each image
    delay_time: np.ndarray, array of floats representing the delay time for each image (rounded to the nearest 1000 femtoseconds)
    '''
    from datetime import datetime

    lab_time = np.zeros(len(im_names), dtype='datetime64[us]')
    fast_scans = np.zeros(len(im_names))
    delay_time = np.zeros(len(im_names))

    for i in range(len(im_names)):
        info = im_names[i].split('_')

        lab_time[i] = datetime(2025, *list(map(int, info[0:5])))
        fast_scans[i] = int(info[6])
        delay_time[i] = np.round(float(info[7][:-1]), -3)  # femtoseconds


    return lab_time, fast_scans, delay_time

# 1. Recuperiamo la lista dei file ignorando i file nascosti del Mac
# Filtriamo: deve finire con .h5 E non deve iniziare con un punto
im_names = [f for f in os.listdir(f_name) if f.endswith('.h5') and not f.startswith('.')]

# 2. Ordiniamo i file alfabeticamente per garantire la corretta sequenza temporale
im_names.sort()

# 3. Ora chiamiamo la funzione get_coordinates sulla lista pulita
lab_time, scan, delay_time = get_coordinates(im_names)

print(f"✅ Successo! Sono stati trovati ed elaborati {len(im_names)} file.")
print(f"Primo file elaborato: {im_names[0]}")

In [ ]:
#Check that everything works fine
i=0
print(im_names[i])
get_coordinates(im_names[i:i+1])

In [ ]:
lab_time, scan, delay_time = get_coordinates(im_names)

In [ ]:
# 1. Prendi le dimensioni REALI dall'array data
n_frames, ny, nx = data.shape

ds = xr.Dataset(
    data_vars=dict(
        im_old=(["lab_time", "y", "x"], data),
    ),

    coords=dict(
        x=(["x"], np.arange(nx)),
        y=(["y"], np.arange(ny)),
        lab_time=(["lab_time"], lab_time),
        # Usiamo 'scan' perché è così che l'hai chiamata nella Cella 11
        scan=(["lab_time"], scan), 
        delay_time=(["lab_time"], delay_time),
    )
)

# 2. Conversione tempi
ds['delay_time'] = ds.delay_time * 1e-3 

# 3. Salvataggio sicuro per Mac e per file di grandi dimensioni
import os

save_path = os.path.join(ds_folder, "ds_no_norm.nc")

# Aggiungi engine='netcdf4' e format='NETCDF4'
ds.to_netcdf(save_path, engine="netcdf4", format="NETCDF4")

print(f"✅ Dataset salvato! Dimensioni: {nx}x{ny} pixel, {n_frames} immagini.")

In [ ]:
ds

### Normalization on counts

Modified this cell due to RAM problems, this dataset is huge so it doesn't compute well with the last cell

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm # Per vedere a che punto siamo ed evitare il panico da "si è bloccato di nuovo?"

# 1. Calcoliamo i conteggi (questo è leggerissimo, produce un array 1D)
counts_sum = ds.im_old.sum(dim=('x', 'y'))
mean_counts = counts_sum.mean().item()

# 2. Prepariamo un "contenitore" vuoto per le nuove immagini
# Questo occupa RAM, ma senza creare i file temporanei giganteschi del broadcasting
im_norm = np.empty_like(ds.im_old.values)

# 3. Riempiamo il contenitore un frame alla volta (Salva-RAM!)
print("Normalizzazione in corso...")
for i in tqdm(range(len(ds.lab_time))):
    # Lavoriamo sui numpy array sottostanti (.values) che sono molto più veloci
    im_norm[i] = (ds.im_old.values[i] / counts_sum.values[i]) * mean_counts

# 4. Assegniamo il risultato al dataset
ds['im'] = (["lab_time", "y", "x"], im_norm)

# 5. Plot di controllo
plt.figure(figsize=(10, 5))
counts_sum.plot(label='sum counts (originali)')
ds.im.sum(dim=('x', 'y')).plot(label='norm counts (normalizzati)')

plt.legend()
plt.title("Confronto Conteggi: Originali vs Normalizzati")
plt.show()

In [ ]:
random_index = np.random.randint(0, len(ds.lab_time))
ds.isel(lab_time=random_index).im.plot()

plt.title('Random image for checking, index %s' % (random_index))

## Animation, to chek everything is OK

In [ ]:
#Check befure running the time consuming animation cell
vmax = 700
vmin = 000

fig, ax = plt.subplots()
im_plot = ds.im.isel(lab_time=0).plot(vmax=vmax, vmin=vmin)
title = ax.set_title(f"Scan: {int(ds.scan.isel(lab_time=0).item())}, delay time: {ds.delay_time.isel(lab_time=0).item():.2f} ps")


In [ ]:
def update(frame):
    im_plot.set_array(ds.im.isel(lab_time=frame).values)
    title.set_text(f"Scan: {int(ds.scan.isel(lab_time=frame).item())}, delay time: {ds.delay_time.isel(lab_time=frame).item():.2f} fs")
    return im_plot, title

In [ ]:
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.animation as animation

# # 1. IMPOSTAZIONI DI SICUREZZA
# # Se hai 1960 immagini, step = 1 le farà tutte (ci vorrà un po' di tempo!).
# # Se vuoi fare un test veloce per vedere se ti piace, metti step = 10 o 20.
# step = 1
# frames_to_plot = range(0, len(ds.lab_time), step)

# # 2. CALCOLO DEL CONTRASTO
# vmin = np.percentile(ds.im.values[0], 1)
# vmax = np.percentile(ds.im.values[0], 99)

# # 3. PREPARAZIONE DEL GRAFICO INIZIALE
# fig, ax = plt.subplots(figsize=(8, 8))
# # Usiamo il numpy array grezzo (.values) per la massima velocità
# cax = ax.imshow(ds.im.values[0], cmap='viridis', vmin=vmin, vmax=vmax)
# plt.colorbar(cax, ax=ax, label='Normalized Counts')

# # --- MODIFICA QUI: Estraiamo sia il delay time che lo scan per il primo frame ---
# delay_t = ds.delay_time.values[0]
# scan_idx = ds.scan.values[0]

# # Impostiamo il titolo iniziale
# title = ax.set_title(f"Scan: {int(scan_idx)} | Delay Time: {delay_t:.2f} ps", fontsize=14)

# # Nascondiamo gli assi per una GIF più pulita (opzionale)
# ax.axis('off')

# # 4. LA FUNZIONE MAGICA CHE NON CONSUMA RAM
# def update(frame_idx):
#     # Sostituiamo solo i dati, non ridisegniamo il plot!
#     cax.set_data(ds.im.values[frame_idx])
    
#     # --- MODIFICA QUI: Aggiorniamo sia il delay time che lo scan ---
#     current_delay = ds.delay_time.values[frame_idx]
#     current_scan = ds.scan.values[frame_idx]
    
#     title.set_text(f"Scan: {int(current_scan)} | Delay Time: {current_delay:.2f} ps")
    
#     return cax, title

# # 5. CREAZIONE DELL'ANIMAZIONE
# print(f"🎬 Creazione GIF in corso su {len(frames_to_plot)} frame")
# ani = animation.FuncAnimation(fig, update, frames=frames_to_plot, blit=True)

# # 6. SALVATAGGIO
# gif_path = os.path.join(ds_folder, "WSe2_dynamics.gif")

# # Se processi molti frame (es. step=1), magari aumenta gli fps a 20 o 30 
# # altrimenti la GIF durerà tantissimo!
# ani.save(gif_path, writer='pillow', fps=15)

# print(f"✅ Finito! GIF salvata con successo in: {gif_path}")

# # Chiudiamo la figura per svuotare la memoria
# plt.close()

### Difference (no normalized images)

Test 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tqdm import tqdm
import os

# --- 1. Creazione sicura di diff_im (Salva-RAM) ---
print("Calcolo dell'immagine di differenza in corso...")
ref_im = ds.im.isel(lab_time=0).values
diff_norm = np.empty_like(ds.im.values)

for i in tqdm(range(len(ds.lab_time))):
    diff_norm[i] = ds.im.values[i] - ref_im

ds['diff_im'] = (["lab_time", "y", "x"], diff_norm)
# --------------------------------------------------

# Check before running the time consuming animation cell
image_to_check = 20
percent = 0.9

#questo prende il massimo della figura, che in questo caso è lo spot centrale (troppo intenso)
#ref_value=ds.im.max()*percent/100

#prendo il massimo in una regione che non ha lo spot centrale
ref_value=ds.im.sel(x=slice(50, 200), y=slice(20, 400)).max()*percent/100
#questo è fatto molto ignorante per fare veloce, prima o poi definire una ROI ecc.

fig, ax = plt.subplots()
im_plot = ds.diff_im.isel(lab_time=image_to_check).plot(vmax=ref_value, vmin=-ref_value, cmap='seismic', rasterized=True,
                      cbar_kwargs={'ticks': [-ref_value, ref_value],
                                   'format': mticker.FixedFormatter([f'{-percent}% ', f'{percent}% ']),
                                   'label': 'of image maximum'})

title = ax.set_title(f"Scan: {int(ds.scan.isel(lab_time=image_to_check).item())}, delay time: {ds.delay_time.isel(lab_time=image_to_check).item():.2f} ps")

# plt.axhline(7.5)

# Save pictures high quality
# fname = os.path.join(plot_folder, 'diff_im.png') # Adattato per il Mac
# plt.savefig(fname, format='png', transparent=True, dpi=300)

plt.show()

In [ ]:
# import os
# import numpy as np
# import matplotlib.pyplot as plt
# import matplotlib.animation as animation
# import matplotlib.ticker as mticker
# from tqdm import tqdm

# # --- 1. CALCOLO DELLA DIFFERENZA (Salva-RAM per il Mac) ---
# print("Calcolo dell'immagine di differenza in corso...")
# ref_im = ds.im.isel(lab_time=0).values # Immagine di riferimento (il primo frame)
# diff_norm = np.empty_like(ds.im.values)

# for i in tqdm(range(len(ds.lab_time))):
#     diff_norm[i] = ds.im.values[i] - ref_im

# ds['diff_im'] = (["lab_time", "y", "x"], diff_norm)


# # --- 2. IMPOSTAZIONI DEL PLOT E CONTRASTO ---
# percent = 5
# # Trova il massimo in una ROI fuori dal centro
# ref_value = ds.im.sel(x=slice(50, 200), y=slice(20, 400)).max().item() * percent / 100

# fig, ax = plt.subplots(figsize=(8, 8))

# # Disegniamo il frame iniziale (lab_time=0)
# im_plot = ds.diff_im.isel(lab_time=0).plot(
#     vmax=ref_value, vmin=-ref_value, cmap='seismic', rasterized=True, ax=ax,
#     add_colorbar=False # Spegniamo la cbar automatica per personalizzarla noi
# )

# # Creiamo la colorbar personalizzata
# cbar = fig.colorbar(im_plot, ax=ax, ticks=[-ref_value, ref_value])
# cbar.ax.set_yticklabels([f'{-percent}% ', f'{percent}% '])
# cbar.set_label('of image maximum')

# # Titolo iniziale
# scan_0 = int(ds.scan.isel(lab_time=0).item())
# delay_0 = ds.delay_time.isel(lab_time=0).item()
# title = ax.set_title(f"Scan: {scan_0} | Delay time: {delay_0:.2f} ps", fontsize=14)

# ax.axis('off') # Nascondiamo gli assi per una GIF più bella


# # --- 3. CREAZIONE DELL'ANIMAZIONE ---
# # Saltiamo qualche frame per non fare una GIF gigantesca e lunghissima (1 per vedere tutta la GIF)
# step = 1
# frames_to_plot = range(0, len(ds.lab_time), step)

# def update(frame_idx):
#     # Aggiorniamo i dati dell'immagine
#     im_plot.set_array(ds.diff_im.isel(lab_time=frame_idx).values)
    
#     # Aggiorniamo il titolo
#     curr_scan = int(ds.scan.isel(lab_time=frame_idx).item())
#     curr_delay = ds.delay_time.isel(lab_time=frame_idx).item()
#     title.set_text(f"Scan: {curr_scan} | Delay time: {curr_delay:.2f} ps")
    
#     return im_plot, title

# print(f"🎬 Creazione GIF differenza su {len(frames_to_plot)} frame in corso...")
# ani = animation.FuncAnimation(fig, update, frames=frames_to_plot, blit=True)

# # Salvataggio usando 'pillow' invece di 'ffmpeg'
# gif_path = os.path.join(plot_folder, 'aligned_images_difference.gif')
# ani.save(gif_path, writer='pillow', fps=10)

# plt.close(fig)
# print(f"✅ GIF differenza creata con successo in: {gif_path}")

## The real data analysis

### order by delay time

In [ ]:
#Unstack the data 
ds_us = ds.set_index(lab_time=('delay_time', 'scan'))

# 864/24 = 36
# This should be 0, otherwise the unstack will not work, which means you didn't put the right coordinates in the get_coordinates function
print('number of duplicates = ', ds_us.im.indexes['lab_time'].duplicated().sum())
ds_us = ds_us.drop_duplicates('lab_time')
ds_us = ds_us.unstack(('lab_time'))

In [ ]:
# Filter out scan 21
# ds_us = ds_us.sel(scan=ds_us.scan != 18.0)

# To remove more than one scan
#ds_us = ds_us.sel(scan=~np.isin(ds_us.scan, [5.0, 12.0, 21.0]))

ds_us

In [ ]:
import hvplot.xarray

vmax = 500
vmin = 000
ds_us.im.isel(delay_time=5, scan=5).hvplot.image(x='x', y='y', clim=(vmin, vmax), cmap ='viridis', aspect='square',
                                       title='To be used for ROI selection')

#  First line

## First spot(1L)

### Rotate the images

In [ ]:
import numpy as np
import xarray as xr
from scipy.ndimage import rotate

# --- DEFINIZIONE COORDINATE FISICHE REALI ---
x0, y0 = 254, 254
x1, y1 = 186, 225
angle_rad = np.arctan2(y1 - y0, x1 - x0)
rot_angle = np.degrees(angle_rad)  

shape_y = ds_us.sizes['y']
shape_x = ds_us.sizes['x']
dummy_img = np.zeros((shape_y, shape_x), dtype=bool)

# La ruotiamo per far calcolare a scipy la nuova cornice
dummy_rotated = rotate(dummy_img, rot_angle, reshape=True)
new_y, new_x = dummy_rotated.shape

def rotate_2d_safe(arr, angle):
    # reshape=True è la chiave! Espande la tela solo quanto basta
    return rotate(arr, angle, reshape=True, order=1, mode='reflect')

# Apply rotation: Salviamo l'output direttamente come DataArray indipendente
im_rotated_temp = xr.apply_ufunc(
    rotate_2d_safe,
    ds_us['im'],
    rot_angle,
    input_core_dims=[['y', 'x'], []],   
    output_core_dims=[['y_new', 'x_new']], 
    vectorize=True,
    dask='parallelized',  
    dask_gufunc_kwargs={'output_sizes': {'y_new': new_y, 'x_new': new_x}}, 
    output_dtypes=[ds_us['im'].dtype]
)

# 1. Riordiniamo usando i nuovi nomi delle dimensioni
im_rotated_temp = im_rotated_temp.transpose('y_new', 'x_new', 'delay_time', 'scan')

# 2. Rinominiamo al volo le dimensioni e salviamo nel DataArray finale
da_rotated = im_rotated_temp.rename({'y_new': 'y', 'x_new': 'x'})

# 3. Assegniamo le coordinate per far felice hvplot
da_rotated = da_rotated.assign_coords(
    x=np.arange(new_x),
    y=np.arange(new_y)
)

In [ ]:
# Il DataArray da_rotated è già pronto dalla cella precedente!
# Andiamo dritti al plot interattivo

vmax = 500
vmin = 0

da_rotated.isel(delay_time=0, scan=5).hvplot.image(
    x='x', y='y', 
    cmap='viridis', clim=(vmin, vmax), aspect='square',
    title='To be used for ROI selection'
)

### Select ROIs

In [ ]:
# Select ROI
roi = da_rotated .sel(y=slice(300, 331), x=slice(129, 458))  #180-230
profile = roi.mean(dim='y')  # dims:(scan, delay_time, y)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

### Fit (1L)

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 200
x_max = 242

x_vals = profile['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 200
x_max = 242

x_vals = profile['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile.sizes['delay_time']
n_scans = profile.sizes['scan']

fit_params = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

### Plots

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))
delay_times = profile['delay_time'].values

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes = fit_params[:, j, 0]        # Extract amplitudes (A) across all delay times for scan j

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized amplitudes

In [ ]:
import plotly.graph_objects as go
import numpy as np


parametri_da_usare = fit_params 

bad_scans = [0,1,2,3]  

# Es: se i primi 4 punti temporali sono prima del t=0 del laser, metti 4
punti_baseline = 4
print(f"scan scartati={bad_scans}")

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

# ==========================================
# --- PLOT FINALE ---
# ==========================================
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Centers

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))
delay_times = profile['delay_time'].values

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers = fit_params[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers = fit_params[:, :, 1]

n_delays, n_scans = centers.shape
mask = np.zeros_like(centers, dtype=bool)

# IL TRUCCO: Chiediamo a Python di includere automaticamente tutti gli scan esistenti
included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers, np.nan)
centers_avg = np.nanmean(masked_centers, axis=1)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=centers_avg,
    mode='lines+markers',
    name='Average Center'
))

fig.update_layout(
    title="Average Center vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Average Center"
)

fig.show()

In [ ]:
# ==========================================
# CALCOLO E PLOT DELLA VARIAZIONE PERCENTUALE (Peak 2)
# ==========================================

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
bad_scans = []

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers2'
n_scans = centers.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg_clean = np.nanmean(centers[:, good_scans], axis=1)

print(f"Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center2_baseline = centers_avg_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers2_avg = ((centers_avg_clean - center2_baseline) / center2_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers2_avg,
    mode='lines+markers',
    name='Δ% Center 2',
    line=dict(color='red', width=3) # Manteniamo il rosso per il Peak 2!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 1) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

# Aggiungiamo la comoda linea tratteggiata sullo zero
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Derivata

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import savgol_filter

idx_delay = 0  
idx_scan = 0   

# Parametri di Smoothing
# window_length: quanti pixel usiamo per mediare (deve essere dispari, es. 5, 7, 9)
# polyorder: ordine del polinomio (2 è lo standard)
window_length = 5
polyorder = 2
# ==========================================

profile_der = profile.isel(delay_time=idx_delay, scan=idx_scan).values
r = np.arange(len(profile_der))

# 2. Operazioni Matematiche
# a) Calcolo della Derivata Prima nuda e cruda
derivative_raw = np.gradient(profile_der) 

# b) Smoothing applicato DOPO la derivata
derivative_sm = savgol_filter(derivative_raw, window_length, polyorder)

# 3. Plot Interattivo con Subplots (Assi X indipendenti!)
fig = make_subplots(
    rows=2, cols=1, 
    shared_xaxes=False, 
    vertical_spacing=0.12, # Aumentato un po' per far spazio alle etichette X del grafico 1
    subplot_titles=(f"Derivata Prima Grezza (Delay {idx_delay}, Scan {idx_scan})", 
                    "Derivata Prima Smoothed (Filtro Savitzky-Golay)")
)

# Traccia 1: Derivata Prima Grezza (Pannello Superiore)
fig.add_trace(go.Scatter(x=r, y=derivative_raw, mode='lines+markers', name='Derivata Grezza', 
                         line=dict(color='gray', width=1), marker=dict(size=4)), row=1, col=1)

# Traccia 2: Derivata Prima Lisciata (Pannello Inferiore)
fig.add_trace(go.Scatter(x=r, y=derivative_sm, mode='lines', name='Derivata Smoothed', 
                         line=dict(color='red', width=3)), row=2, col=1)

# Linee dello zero per allineare l'occhio
fig.add_hline(y=0, line_dash="dash", line_color="black", row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="black", row=2, col=1)

fig.update_layout(
    height=800, width=900, 
    hovermode="x unified",
    template="plotly_white",
    title_text="Tool Diagnostico: Effetto dello Smoothing sulla Derivata"
)

# Etichette assi (Ora per entrambi i grafici)
fig.update_xaxes(title_text="Raggio (pixel)", row=1, col=1)
fig.update_xaxes(title_text="Raggio (pixel)", row=2, col=1)
fig.update_yaxes(title_text="Δ Intensità (Grezza)", row=1, col=1)
fig.update_yaxes(title_text="Δ Intensità (Smoothed)", row=2, col=1)

fig.show()

In [ ]:
import numpy as np
import xarray as xr
from scipy.signal import savgol_filter

window_length = 5
polyorder = 2

# 1. Estraiamo la matrice 3D grezza (delay_time, scan)
data_matrix = profile.values

# 2. Calcoliamo la derivata prima lungo l'asse del raggio
derivative_raw = np.gradient(data_matrix, axis=-1)

# 3. Applichiamo il filtro di Savitzky-Golay sempre lungo l'asse del raggio
derivative_smoothed = savgol_filter(derivative_raw, window_length, polyorder, axis=-1)

# 4. Ricreiamo un xarray DataArray per mantenere la comodità delle coordinate
deriv_sm = xr.DataArray(
    derivative_smoothed,
    dims=profile.dims,      # Mantiene ('delay_time', 'scan', 'radius')
    coords=profile.coords,  # Mantiene i valori esatti di tempi, scan e pixel
    name="Smoothed_Derivative"
)

print("✅ Calcolo completato! Risultato salvato in 'deriv_sm'.")

In [ ]:
import numpy as np
import plotly.graph_objects as go

x1=70
x2=94

idx_baseline = 0
bad_scans = []
# ==========================================

n_delays = deriv_sm.sizes['delay_time']
n_scans = deriv_sm.sizes['scan']
delays = deriv_sm['delay_time'].values

# PRENDIAMO I DATI GIÀ FILTRATI DAL TUO XARRAY
deriv_values_sm = deriv_sm.values
peak_amplitude = np.zeros((n_delays, n_scans))


for i in range(n_delays):
    for j in range(n_scans):
        # Prendiamo la derivata già lisciata
        ydata_sm = deriv_sm.isel(delay_time=i, scan=j).values
        
        # Estrai Ampiezza
        roi_y_sm = ydata_sm[x1:x2]
        peak_amplitude[i, j] = np.abs(np.max(roi_y_sm) - np.min(roi_y_sm))

# --- NORMALIZZAZIONE AMPIEZZA (I / I0) ---
good_scans = [s for s in range(n_scans) if s not in bad_scans]
amp_t0 = peak_amplitude[idx_baseline, :]
norm_amplitude = peak_amplitude / amp_t0
mean_amp = np.mean(norm_amplitude[:, good_scans], axis=1)

print("✅ Estrazione Ampiezza completata!")

# --- PLOT AMPIEZZA ---
fig_amp = go.Figure()
fig_amp.add_trace(go.Scatter(
    x=delays, y=mean_amp, mode='lines+markers', 
    line=dict(color='blue', width=3), name=f'Media ({len(good_scans)} scan)'
))
fig_amp.add_hline(y=1, line_dash="dash", line_color="black")

fig_amp.update_layout(
    title=f"Dinamica I/I₀ (Metodo Derivata Lisciata) - Picco [{x1}:{x2}]",
    xaxis_title="Delay Time (ps)", yaxis_title="I / I₀",
    hovermode="x unified", template="plotly_white", height=450, width=900
)
fig_amp.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

x1=70
x2=94
q = 3

idx_baseline = 0
bad_scans = []

n_delays = deriv_sm.sizes['delay_time']
n_scans = deriv_sm.sizes['scan']
delays = deriv_sm['delay_time'].values

deriv_values_sm = deriv_sm.values  
peak_position = np.zeros((n_delays, n_scans))
for i in range(n_delays):
    for j in range(n_scans):
        # Prendiamo la derivata già lisciata
        ydata_sm = deriv_sm.isel(delay_time=i, scan=j).values
        
        # 1. Trova il minimo locale
        roi_y_sm = ydata_sm[x1:x2]
        local_min_idx = np.argmin(roi_y_sm)
        xmin = x1 + local_min_idx
        
        # 2. Finestra ± q
        xvect = np.arange(xmin - q, xmin + q + 1)
        window_y_sm = ydata_sm[xmin - q : xmin + q + 1]
        
        # 3. Centro di massa
        peak_position[i, j] = np.sum(xvect * window_y_sm) / np.sum(window_y_sm)

# --- NORMALIZZAZIONE POSIZIONE (Δμ / μ0 in %) ---
good_scans = [s for s in range(n_scans) if s not in bad_scans]
pos_t0 = peak_position[idx_baseline, :]
delta_pos = ((peak_position - pos_t0) / pos_t0) * 100
mean_pos = np.mean(delta_pos[:, good_scans], axis=1)

print("✅ Estrazione Posizione completata!")

# --- PLOT POSIZIONE ---
fig_pos = go.Figure()
fig_pos.add_trace(go.Scatter(
    x=delays, y=mean_pos, mode='lines+markers', 
    line=dict(color='red', width=3), name=f'Media ({len(good_scans)} scan)'
))
fig_pos.add_hline(y=0, line_dash="dash", line_color="black")

fig_pos.update_layout(
    title=f"Evoluzione Δμ/μ₀ (Centro di Massa Derivata Lisciata, q={q})",
    xaxis_title="Delay Time (ps)", yaxis_title="Δμ / μ₀ (%)",
    hovermode="x unified", template="plotly_white", height=450, width=900
)
fig_pos.show()

## Second spot (1R)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 349
x_max = 382

x_vals = profile['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 349
x_max = 382
x_vals = profile['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile.sizes['delay_time']
n_scans = profile.sizes['scan']

fit_params2 = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params2[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))
delay_times = profile['delay_time'].values

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes2 = fit_params2[:, j, 0]        # Extract amplitudes (A) across all delay times for scan j

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes2,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized amplitudes

In [ ]:
import plotly.graph_objects as go
import numpy as np

# ==========================================
# --- IMPOSTAZIONI ---
# ==========================================
# 1. Scegli quale array di parametri usare (fit_params o fit_params1L)
parametri_da_usare = fit_params2

# 2. Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [0,1,2,3] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
# Es: se i primi 4 punti temporali sono prima del t=0 del laser, metti 4
punti_baseline = 4
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

# ==========================================
# --- PLOT FINALE ---
# ==========================================
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

Sum of amplitudes

In [ ]:
import plotly.graph_objects as go
import numpy as np

# 1. Inserisci i DUE array di parametri dei due spot simmetrici
parametri_spot_A = fit_params  
parametri_spot_B = fit_params2 

bad_scans = [0,1,2,3] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values
n_scans = parametri_spot_A.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_A = parametri_spot_A[:, good_scans, 0]
amp_B = parametri_spot_B[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_A = np.nanmean(amp_A, axis=1)
mean_pura_B = np.nanmean(amp_B, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_A + mean_pura_B

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale


fig = go.Figure()

#singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_A / np.nanmean(mean_pura_A[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot A'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_B / np.nanmean(mean_pura_B[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot B'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (1+2) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Centers

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))
delay_times = profile['delay_time'].values

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers2 = fit_params2[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers2,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers2 = fit_params2[:, :, 1]
n_delays, n_scans = centers2.shape
centers2_avg = np.nanmean(centers2, axis=1)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=centers2_avg,
    mode='lines+markers',
    name='Average Center'
))

fig.update_layout(
    title="Average Center vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Average Center"
)

fig.show()

In [ ]:
bad_scans = [] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers2'
n_scans = centers2.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg2_clean = np.nanmean(centers2[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 2. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center2_baseline = centers_avg2_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers2_avg = ((centers_avg2_clean - center2_baseline) / center2_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers2_avg,
    mode='lines+markers',
    name='Δ% Center 2',
    line=dict(color='red', width=3) # Manteniamo il rosso per il Peak 2!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 2) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

# Aggiungiamo la comoda linea tratteggiata sullo zero
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

Center difference

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Inserisci qui l'elenco degli scan che vuoi includere nel calcolo.
# - Solo lo scan 0: [0]
# - Più scan specifici: [0, 1, 2, 4] (salta il 3)
# - Tutti gli scan fino al 5: list(range(6))
scans_to_include = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31]


# 1. Filtriamo le matrici originali prendendo SOLO gli scan scelti
centers_sel_avg = np.nanmean(centers[:, scans_to_include], axis=1)
centers2_sel_avg = np.nanmean(centers2[:, scans_to_include], axis=1)

# 2. Calcoliamo la distanza assoluta tra i due spot (D) per ogni delay time
distanza_centri = np.abs(centers2_sel_avg - centers_sel_avg)

# 3. CALCOLO DELLA BASELINE
# Facciamo la media di TUTTI i tempi negativi per un riferimento stabile
baseline_indices = np.where(delay_times < -200)[0] # Seleziona i punti pre-pompaggio
if len(baseline_indices) == 0:
    baseline_indices = np.arange(5)

dist_baseline_D0 = np.mean(distanza_centri[baseline_indices])

# 4. Calcoliamo la Variazione Percentuale
# Formula: ((D(t) - D0) / D0) * 100
variazione_perc = ((distanza_centri - dist_baseline_D0) / dist_baseline_D0) * 100


fig_dist = go.Figure()

fig_dist.add_trace(go.Scatter(
    x=delay_times,
    y=variazione_perc,
    mode='lines+markers',
    line=dict(color='green', width=3),
    name=f'Δ Distanza (%) - Scan {scans_to_include}'
))

fig_dist.add_hline(y=0, line_dash="dash", line_color="black")

fig_dist.update_layout(
    title=f"Relative distance 1-2 vs Delay time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Relative distance (%)",
    hovermode="x unified",
    template="plotly_white",
    height=450, width=900
)

fig_dist.show()

## Third spot (2L)

In [ ]:
x_min = 123
x_max = 156
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
x_fit = x_vals[fit_mask]

ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 123
x_max = 156

x_vals = profile['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile.sizes['delay_time']
n_scans = profile.sizes['scan']

fit_params3 = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params3[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes3 = fit_params3[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes3,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

#### Normalized amplitudes

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params3

# 2. Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans=[0,1,2,3]

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
# Es: se i primi 4 punti temporali sono prima del t=0 del laser, metti 4
punti_baseline = 4
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

# ==========================================
# --- PLOT FINALE ---
# ==========================================
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

#### Center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers3 = fit_params3[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers3,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers3 = fit_params3[:, :, 1]

n_delays, n_scans = centers3.shape
mask = np.zeros_like(centers3, dtype=bool)

# IL TRUCCO: Chiediamo a Python di includere automaticamente tutti gli scan esistenti
included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers3, np.nan)
centers_avg3 = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [0,1,2,3] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers3  '
n_scans = centers3.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg3_clean = np.nanmean(centers3[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 3. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center3_baseline = centers_avg3_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers3_avg = ((centers_avg3_clean - center3_baseline) / center3_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers3_avg,
    mode='lines+markers',
    name='Δ% Center 3',
    line=dict(color='red', width=3) # Manteniamo il rosso per il Peak 3!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 3) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

## Fourth spot (2R)

In [ ]:
x_min = 421
x_max = 457
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
x_fit = x_vals[fit_mask]

ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 421
x_max = 457

x_vals = profile['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile.sizes['delay_time']
n_scans = profile.sizes['scan']

fit_params4 = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params4[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes4 = fit_params4[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes4,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

#### Normalized amplitudes

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params4

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

# ==========================================
# --- PLOT FINALE ---
# ==========================================
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

Summed Ampltudes

In [ ]:
import plotly.graph_objects as go
import numpy as np
import warnings

# 1. Inserisci i DUE array di parametri dei due spot simmetrici
parametri_spot_C = fit_params3  
parametri_spot_D = fit_params4

bad_scans = [] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values

n_scans = parametri_spot_C.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_C = parametri_spot_C[:, good_scans, 0]
amp_D = parametri_spot_D[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_C = np.nanmean(amp_C, axis=1)
mean_pura_D = np.nanmean(amp_D, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_C + mean_pura_D

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale

fig = go.Figure()

# singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_C / np.nanmean(mean_pura_C[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot C'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_D / np.nanmean(mean_pura_D[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot D'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (3+4) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

#### Peak centers

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers4 = fit_params4[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers4,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers4 = fit_params4[:, :, 1]

n_delays, n_scans = centers4.shape
mask = np.zeros_like(centers4, dtype=bool)

# IL TRUCCO: Chiediamo a Python di includere automaticamente tutti gli scan esistenti
included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers4, np.nan)
centers_avg4 = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [0,1,2,3] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers4  '
n_scans = centers4.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg4_clean = np.nanmean(centers4[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 4. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center4_baseline = centers_avg4_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers4_avg = ((centers_avg4_clean - center4_baseline) / center4_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers4_avg,
    mode='lines+markers',
    name='Δ% Center 4',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 4!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 4) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

Difference


In [ ]:
import numpy as np
import plotly.graph_objects as go

# Inserisci qui l'elenco degli scan che vuoi includere nel calcolo.
# - Solo lo scan 0: [0]
# - Più scan specifici: [0, 1, 2, 4] (salta il 3)
# - Tutti gli scan fino al 5: list(range(6))
scans_to_include = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31]


# 1. Filtriamo le matrici originali prendendo SOLO gli scan scelti
centers3_sel_avg = np.nanmean(centers3[:, scans_to_include], axis=1)
centers4_sel_avg = np.nanmean(centers4[:, scans_to_include], axis=1)

# 2. Calcoliamo la distanza assoluta tra i due spot (D) per ogni delay time
distanza_centri = np.abs(centers4_sel_avg - centers3_sel_avg)

# 3. CALCOLO DELLA BASELINE
# Facciamo la media di TUTTI i tempi negativi per un riferimento stabile
baseline_indices = np.where(delay_times < -200)[0] # Seleziona i punti pre-pompaggio
if len(baseline_indices) == 0:
    baseline_indices = np.arange(5)

dist_baseline_D0 = np.mean(distanza_centri[baseline_indices])

# 4. Calcoliamo la Variazione Percentuale
# Formula: ((D(t) - D0) / D0) * 100
variazione_perc = ((distanza_centri - dist_baseline_D0) / dist_baseline_D0) * 100


fig_dist = go.Figure()

fig_dist.add_trace(go.Scatter(
    x=delay_times,
    y=variazione_perc,
    mode='lines+markers',
    line=dict(color='green', width=3),
    name=f'Δ Distanza (%) - Scan {scans_to_include}'
))

fig_dist.add_hline(y=0, line_dash="dash", line_color="black")

fig_dist.update_layout(
    title=f"Relative distance 3-4 vs Delay time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Relative distance (%)",
    hovermode="x unified",
    template="plotly_white",
    height=450, width=900
)
fig_dist.show()

## Fifth spot (3L)

In [ ]:
x_min = 65
x_max = 116
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
x_fit = x_vals[fit_mask]

ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 65
x_max = 116

x_vals = profile['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile.sizes['delay_time']
n_scans = profile.sizes['scan']

fit_params5 = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params5[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes5 = fit_params5[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes5,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

#### Normalized Amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params5

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

# ==========================================
# --- PLOT FINALE ---
# ==========================================
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

#### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers5 = fit_params5[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers5,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers5 = fit_params5[:, :, 1]

n_delays, n_scans = centers5.shape
mask = np.zeros_like(centers5, dtype=bool)

# IL TRUCCO: Chiediamo a Python di includere automaticamente tutti gli scan esistenti
included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers5, np.nan)
centers_avg5 = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers5  '
n_scans = centers5.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg5_clean = np.nanmean(centers5[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 5. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center5_baseline = centers_avg5_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers5_avg = ((centers_avg5_clean - center5_baseline) / center5_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers5_avg,
    mode='lines+markers',
    name='Δ% Center 5',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 5!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 5) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

## Sixth spot (3R)

In [ ]:
x_min = 491
x_max = 536
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
x_fit = x_vals[fit_mask]

ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 491
x_max = 536

x_vals = profile['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile.sizes['delay_time']
n_scans = profile.sizes['scan']

fit_params6 = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params6[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes6 = fit_params6[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes6,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

#### Normalized amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params6

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

Summed amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np
import warnings

# 1. Inserisci i DUE array di parametri dei due spot simmetrici
parametri_spot_E = fit_params5 
parametri_spot_F = fit_params6

bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values

n_scans = parametri_spot_E.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_E = parametri_spot_E[:, good_scans, 0]
amp_F = parametri_spot_F[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_E = np.nanmean(amp_E, axis=1)
mean_pura_F = np.nanmean(amp_F, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_E + mean_pura_F

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale

fig = go.Figure()

# singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_E / np.nanmean(mean_pura_E[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot E'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_F / np.nanmean(mean_pura_F[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot F'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (5+6) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

#### peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers6 = fit_params6[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers6,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers6 = fit_params6[:, :, 1]

n_delays, n_scans = centers6.shape
mask = np.zeros_like(centers6, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers6, np.nan)
centers_avg6 = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers6  '
n_scans = centers6.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg6_clean = np.nanmean(centers6[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 6. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center6_baseline = centers_avg6_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers6_avg = ((centers_avg6_clean - center6_baseline) / center6_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers6_avg,
    mode='lines+markers',
    name='Δ% Center 6',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 6!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 6) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

difference


In [ ]:
import numpy as np
import plotly.graph_objects as go

# Inserisci qui l'elenco degli scan che vuoi includere nel calcolo.
# - Solo lo scan 0: [0]
# - Più scan specifici: [0, 1, 2, 4] (salta il 3)
# - Tutti gli scan fino al 5: list(range(6))
scans_to_include = [0,1,2,3]


# 1. Filtriamo le matrici originali prendendo SOLO gli scan scelti
centers5_sel_avg = np.nanmean(centers5[:, scans_to_include], axis=1)
centers6_sel_avg = np.nanmean(centers6[:, scans_to_include], axis=1)

# 2. Calcoliamo la distanza assoluta tra i due spot (D) per ogni delay time
distanza_centri = np.abs(centers6_sel_avg - centers5_sel_avg)

# 3. CALCOLO DELLA BASELINE
# Facciamo la media di TUTTI i tempi negativi per un riferimento stabile
baseline_indices = np.where(delay_times < -200)[0] # Seleziona i punti pre-pompaggio
if len(baseline_indices) == 0:
    baseline_indices = np.arange(5)

dist_baseline_D0 = np.mean(distanza_centri[baseline_indices])

# 4. Calcoliamo la Variazione Percentuale
# Formula: ((D(t) - D0) / D0) * 100
variazione_perc = ((distanza_centri - dist_baseline_D0) / dist_baseline_D0) * 100


fig_dist = go.Figure()

fig_dist.add_trace(go.Scatter(
    x=delay_times,
    y=variazione_perc,
    mode='lines+markers',
    line=dict(color='green', width=3),
    name=f'Δ Distanza (%) - Scan {scans_to_include}'
))

fig_dist.add_hline(y=0, line_dash="dash", line_color="black")

fig_dist.update_layout(
    title=f"Relative distance 5-6 vs Delay time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Relative distance (%)",
    hovermode="x unified",
    template="plotly_white",
    height=450, width=900
)
fig_dist.show()

# Second line

In [ ]:
import hvplot.xarray

vmax = 500
vmin = 000
ds_us.im.isel(delay_time=5, scan=5).hvplot.image(x='x', y='y', clim=(vmin, vmax), cmap ='viridis', aspect='square',
                                       title='To be used for ROI selection')

## First spot

### Rotate the images

In [ ]:
import numpy as np
import xarray as xr
from scipy.ndimage import rotate

x0, y0 = 254, 254
x1, y1 = 312, 201
angle_rad = np.arctan2(y1 - y0, x1 - x0)
rot_angle = np.degrees(angle_rad)  

shape_y = ds_us.sizes['y']
shape_x = ds_us.sizes['x']
dummy_img = np.zeros((shape_y, shape_x), dtype=bool)

# La ruotiamo per far calcolare a scipy la nuova cornice
dummy_rotated = rotate(dummy_img, rot_angle, reshape=True)
new_y, new_x = dummy_rotated.shape

def rotate_2d_safe(arr, angle):
    return rotate(arr, angle, reshape=True, order=1, mode='reflect')

# Apply rotation. Il risultato è già un Xarray DataArray!
da_rotated_temp = xr.apply_ufunc(     
    rotate_2d_safe,
    ds_us['im'],
    rot_angle,
    input_core_dims=[['y', 'x'], []],   
    output_core_dims=[['y_dir2', 'x_dir2']], 
    vectorize=True,
    dask='parallelized',  
    dask_gufunc_kwargs={'output_sizes': {'y_dir2': new_y, 'x_dir2': new_x}}, 
    output_dtypes=[ds_us['im'].dtype]
)

# 1. Riordiniamo le dimensioni
da_rotated_temp = da_rotated_temp.transpose('y_dir2', 'x_dir2', 'delay_time', 'scan')

# 2. Rinominiamo le dimensioni al volo in 'y' e 'x'
da_rotated_2 = da_rotated_temp.rename({'y_dir2': 'y', 'x_dir2': 'x'})

# 3. Assegniamo le coordinate numeriche per accontentare hvPlot
da_rotated_2 = da_rotated_2.assign_coords(
    x=np.arange(new_x),
    y=np.arange(new_y)
)


In [ ]:
vmax = 500
vmin = 0

da_rotated_2.isel(delay_time=5, scan=5).hvplot.image(
    x='x', y='y', 
    cmap='viridis', clim=(vmin, vmax), aspect='square',
    title='To be used for ROI selection'
)

Select ROIs

In [ ]:
roi2 = da_rotated_2.sel(y=slice(344, 374), x=slice(154, 530)) 
profile2 = roi2.mean(dim='y')

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile2['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile2.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile2.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

### Fit (1L)

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 249
x_max = 288

x_vals = profile2['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile2.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile2.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 249
x_max = 288

x_vals = profile2['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile2.sizes['delay_time']
n_scans = profile2.sizes['scan']

fit_params2_1L = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile2.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params2_1L[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes2_1L = fit_params2_1L[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes2_1L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized Amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params2_1L

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers2_1L = fit_params2_1L[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers2_1L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers2_1L = fit_params2_1L[:, :, 1]

n_delays, n_scans = centers2_1L.shape
mask = np.zeros_like(centers2_1L, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers2_1L, np.nan)
centers_avg2_1L = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers2_1L  '
n_scans = centers2_1L.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg2_1L_clean = np.nanmean(centers2_1L[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 2_1L. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center2_1L_baseline = centers_avg2_1L_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers2_1L_avg = ((centers_avg2_1L_clean - center2_1L_baseline) / center2_1L_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers2_1L_avg,
    mode='lines+markers',
    name='Δ% Center 2_1L',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 2_1L!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 2_1L) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

## Second spot (1R)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile2['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile2.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile2.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 407
x_max = 449

x_vals = profile2['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile2.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile2.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 402
x_max = 439

x_vals = profile2['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile2.sizes['delay_time']
n_scans = profile2.sizes['scan']

fit_params2_1R = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile2.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params2_1R[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes2_1R = fit_params2_1R[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes2_1R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params2_1R

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

Summed

In [ ]:
import plotly.graph_objects as go
import numpy as np
import warnings

# 1. Inserisci i DUE array di parametri dei due spot simmetrici
parametri_spot_2_A = fit_params2_1L
parametri_spot_2_B = fit_params2_1R

bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values

n_scans = parametri_spot_2_A.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_2_A = parametri_spot_2_A[:, good_scans, 0]
amp_2_B = parametri_spot_2_B[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_2_A = np.nanmean(amp_2_A, axis=1)
mean_pura_2_B = np.nanmean(amp_2_B, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_2_A + mean_pura_2_B

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale

fig = go.Figure()

# singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_2_A / np.nanmean(mean_pura_2_A[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot 2A'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_2_B / np.nanmean(mean_pura_2_B[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot 2B'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (7+8) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers2_1R = fit_params2_1R[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers2_1R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers2_1R = fit_params2_1R[:, :, 1]

n_delays, n_scans = centers2_1R.shape
mask = np.zeros_like(centers2_1R, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers2_1R, np.nan)
centers_avg2_1R = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers2_1R  '
n_scans = centers2_1R.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg2_1R_clean = np.nanmean(centers2_1R[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 2_1R. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center2_1R_baseline = centers_avg2_1R_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers2_1R_avg = ((centers_avg2_1R_clean - center2_1R_baseline) / center2_1R_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers2_1R_avg,
    mode='lines+markers',
    name='Δ% Center 2_1R',
    line=dict(color='red', width=3) # Manteniamo il rosso per il Peak 2_1R!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 2_1R) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

difference

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Inserisci qui l'elenco degli scan che vuoi includere nel calcolo.
# - Solo lo scan 0: [0]
# - Più scan specifici: [0, 1, 2, 4] (salta il 3)
# - Tutti gli scan fino al 5: list(range(6))
scans_to_include = [0,1,2,3]


# 1. Filtriamo le matrici originali prendendo SOLO gli scan scelti
centers2_1L_sel_avg = np.nanmean(centers2_1L[:, scans_to_include], axis=1)
centers2_1R_sel_avg = np.nanmean(centers2_1R[:, scans_to_include], axis=1)

# 2. Calcoliamo la distanza assoluta tra i due spot (D) per ogni delay time
distanza_centri = np.abs(centers2_1R_sel_avg - centers2_1L_sel_avg)

# 3. CALCOLO DELLA BASELINE
# Facciamo la media di TUTTI i tempi negativi per un riferimento stabile
baseline_indices = np.where(delay_times < -200)[0] # Seleziona i punti pre-pompaggio
if len(baseline_indices) == 0:
    baseline_indices = np.arange(5)

dist_baseline_D0 = np.mean(distanza_centri[baseline_indices])

# 4. Calcoliamo la Variazione Percentuale
# Formula: ((D(t) - D0) / D0) * 100
variazione_perc = ((distanza_centri - dist_baseline_D0) / dist_baseline_D0) * 100


fig_dist = go.Figure()

fig_dist.add_trace(go.Scatter(
    x=delay_times,
    y=variazione_perc,
    mode='lines+markers',
    line=dict(color='green', width=3),
    name=f'Δ Distanza (%) - Scan {scans_to_include}'
))

fig_dist.add_hline(y=0, line_dash="dash", line_color="black")

fig_dist.update_layout(
    title=f"Relative distance 7-8 vs Delay time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Relative distance (%)",
    hovermode="x unified",
    template="plotly_white",
    height=450, width=900
)
fig_dist.show()

## Third spot (2L)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile2['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile2.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile2.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 125
x_max = 169

x_vals = profile2['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile2.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile2.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 125
x_max = 169

x_vals = profile2['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile2.sizes['delay_time']
n_scans = profile2.sizes['scan']

fit_params2_2L = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile2.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params2_2L[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes2_2L = fit_params2_2L[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes2_2L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized Amplitude 

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params2_2L

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers2_2L = fit_params2_2L[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers2_2L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers2_2L = fit_params2_2L[:, :, 1]

n_delays, n_scans = centers2_2L.shape
mask = np.zeros_like(centers2_2L, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers2_2L, np.nan)
centers_avg2_2L = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers2_2L  '
n_scans = centers2_2L.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg2_2L_clean = np.nanmean(centers2_2L[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 2_2L. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center2_2L_baseline = centers_avg2_2L_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers2_2L_avg = ((centers_avg2_2L_clean - center2_2L_baseline) / center2_2L_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers2_2L_avg,
    mode='lines+markers',
    name='Δ% Center 2_2L',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 2_2L!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 2_2L) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

## Fourth spot (2R)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile2['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile2.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile2.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 490
x_max = 538

x_vals = profile2['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile2.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile2.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 490
x_max = 538

x_vals = profile2['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile2.sizes['delay_time']
n_scans = profile2.sizes['scan']

fit_params2_2R = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile2.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params2_2R[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes2_2R = fit_params2_2R[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes2_2R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params2_2R

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

summed

In [ ]:
import plotly.graph_objects as go
import numpy as np
import warnings

# 1. Inserisci i DUE array di parametri dei due spot simmetrici
parametri_spot_2_C = fit_params2_2L
parametri_spot_2_D = fit_params2_2R

bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values

n_scans = parametri_spot_2_A.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_2_C = parametri_spot_2_C[:, good_scans, 0]
amp_2_D = parametri_spot_2_D[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_2_C = np.nanmean(amp_2_C, axis=1)
mean_pura_2_D = np.nanmean(amp_2_D, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_2_C + mean_pura_2_D

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale

fig = go.Figure()

# singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_2_A / np.nanmean(mean_pura_2_A[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot 2A'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_2_B / np.nanmean(mean_pura_2_B[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot 2B'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (9+10) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers2_2R = fit_params2_2R[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers2_2R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers2_2R = fit_params2_2R[:, :, 1]

n_delays, n_scans = centers2_2R.shape
mask = np.zeros_like(centers2_2R, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers2_2R, np.nan)
centers_avg2_2R = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers2_2R  '
n_scans = centers2_2R.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg2_2R_clean = np.nanmean(centers2_2R[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 2_2R. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center2_2R_baseline = centers_avg2_2R_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers2_2R_avg = ((centers_avg2_2R_clean - center2_2R_baseline) / center2_2R_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers2_2R_avg,
    mode='lines+markers',
    name='Δ% Center 2_2R',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 2_2R!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 2_2R) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

Difference

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Inserisci qui l'elenco degli scan che vuoi includere nel calcolo.
# - Solo lo scan 0: [0]
# - Più scan specifici: [0, 1, 2, 4] (salta il 3)
# - Tutti gli scan fino al 5: list(range(6))
scans_to_include = [0,1,2,3]

# 1. Filtriamo le matrici originali prendendo SOLO gli scan scelti
centers2_2L_sel_avg = np.nanmean(centers2_2L[:, scans_to_include], axis=1)
centers2_2R_sel_avg = np.nanmean(centers2_2R[:, scans_to_include], axis=1)

# 2. Calcoliamo la distanza assoluta tra i due spot (D) per ogni delay time
distanza_centri = np.abs(centers2_2R_sel_avg - centers2_2L_sel_avg)

# 3. CALCOLO DELLA BASELINE
# Facciamo la media di TUTTI i tempi negativi per un riferimento stabile
baseline_indices = np.where(delay_times < -200)[0] # Seleziona i punti pre-pompaggio
if len(baseline_indices) == 0:
    baseline_indices = np.arange(5)

dist_baseline_D0 = np.mean(distanza_centri[baseline_indices])

# 4. Calcoliamo la Variazione Percentuale
# Formula: ((D(t) - D0) / D0) * 100
variazione_perc = ((distanza_centri - dist_baseline_D0) / dist_baseline_D0) * 100


fig_dist = go.Figure()

fig_dist.add_trace(go.Scatter(
    x=delay_times,
    y=variazione_perc,
    mode='lines+markers',
    line=dict(color='green', width=3),
    name=f'Δ Distanza (%) - Scan {scans_to_include}'
))

fig_dist.add_hline(y=0, line_dash="dash", line_color="black")

fig_dist.update_layout(
    title=f"Relative distance 9-10 vs Delay time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Relative distance (%)",
    hovermode="x unified",
    template="plotly_white",
    height=450, width=900
)

fig_dist.show()

# Third line

In [ ]:
import hvplot.xarray

vmax = 100
vmin = 000
ds_us.im.isel(delay_time=5, scan=5).hvplot.image(x='x', y='y', clim=(vmin, vmax), cmap ='viridis', aspect='square',
                                       title='To be used for ROI selection')

## Rotate the image

In [ ]:
import numpy as np
import xarray as xr
from scipy.ndimage import rotate

# Le tue nuove coordinate (il centro x0, y0 rimane uguale)
x0, y0 = 261, 268 
x1, y1 = 283, 181
angle_rad = np.arctan2(y1 - y0, x1 - x0)
rot_angle = np.degrees(angle_rad)  

shape_y = ds_us.sizes['y']
shape_x = ds_us.sizes['x']
dummy_img = np.zeros((shape_y, shape_x), dtype=bool)

# Calcoliamo la nuova cornice
dummy_rotated = rotate(dummy_img, rot_angle, reshape=True)
new_y, new_x = dummy_rotated.shape

def rotate_2d_safe(arr, angle):
    return rotate(arr, angle, reshape=True, order=1, mode='reflect')

# Apply rotation: usiamo 'dir3' per le dimensioni temporanee
da_rotated_temp_3 = xr.apply_ufunc(     
    rotate_2d_safe,
    ds_us['im'],
    rot_angle,
    input_core_dims=[['y', 'x'], []],   
    output_core_dims=[['y_dir3', 'x_dir3']],
    vectorize=True,
    dask='parallelized',  
    dask_gufunc_kwargs={'output_sizes': {'y_dir3': new_y, 'x_dir3': new_x}},
    output_dtypes=[ds_us['im'].dtype]
)

# 1. Riordiniamo le dimensioni
da_rotated_temp_3 = da_rotated_temp_3.transpose('y_dir3', 'x_dir3', 'delay_time', 'scan')

da_rotated_3 = da_rotated_temp_3.rename({'y_dir3': 'y', 'x_dir3': 'x'})

# 3. Assegniamo le coordinate numeriche
da_rotated_3 = da_rotated_3.assign_coords(
    x=np.arange(new_x),
    y=np.arange(new_y)
)

In [ ]:
vmax = 100
vmin = 0

da_rotated_3.isel(delay_time=0, scan=5).hvplot.image(
    x='x', y='y', 
    cmap='viridis', clim=(vmin, vmax), aspect='square',
    title='To be used for ROI selection'
)

In [ ]:
roi3 = da_rotated_3.sel(y=slice(300, 333), x=slice(70, 475)) 
profile3 = roi3.mean(dim='y')

## First spot (1L)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile3['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile3.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile3.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 170
x_max = 207

x_vals = profile3['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile3.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile3.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 170
x_max = 207

x_vals = profile3['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile3.sizes['delay_time']
n_scans = profile3.sizes['scan']

fit_params3_1L = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile3.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params3_1L[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes3_1L = fit_params3_1L[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes3_1L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params3_1L

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers3_1L = fit_params3_1L[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers3_1L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers3_1L = fit_params3_1L[:, :, 1]

n_delays, n_scans = centers3_1L.shape
mask = np.zeros_like(centers3_1L, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers3_1L, np.nan)
centers_avg3_1L = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers3_1L  '
n_scans = centers3_1L.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg3_1L_clean = np.nanmean(centers3_1L[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 3_1L. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center3_1L_baseline = centers_avg3_1L_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers3_1L_avg = ((centers_avg3_1L_clean - center3_1L_baseline) / center3_1L_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers3_1L_avg,
    mode='lines+markers',
    name='Δ% Center 3_1L',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 3_1L!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 3_1L) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

## Second spot (1R)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile3['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile3.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile3.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 348
x_max = 375

x_vals = profile3['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile3.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile3.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 348
x_max = 375

x_vals = profile3['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile3.sizes['delay_time']
n_scans = profile3.sizes['scan']

fit_params3_1R = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile3.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params3_1R[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes3_1R = fit_params3_1R[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes3_1R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params3_1R

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

summed

In [ ]:
import plotly.graph_objects as go
import numpy as np
import warnings

# 1. Inserisci i DUE array di parametri dei due spot simmetrici
parametri_spot_3_A = fit_params3_1L
parametri_spot_3_B = fit_params3_1R

bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values

n_scans = parametri_spot_3_A.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_3_A = parametri_spot_3_A[:, good_scans, 0]
amp_3_B = parametri_spot_3_B[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_3_A = np.nanmean(amp_3_A, axis=1)
mean_pura_3_B = np.nanmean(amp_3_B, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_3_A + mean_pura_3_B

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale

fig = go.Figure()

# singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_3_A / np.nanmean(mean_pura_3_A[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot 3A'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_3_B / np.nanmean(mean_pura_3_B[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot 3B'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (11+12) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers3_1R = fit_params3_1R[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers3_1R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers3_1R = fit_params3_1R[:, :, 1]

n_delays, n_scans = centers3_1R.shape
mask = np.zeros_like(centers3_1R, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers3_1R, np.nan)
centers_avg3_1R = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers3_1R  '
n_scans = centers3_1R.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg3_1R_clean = np.nanmean(centers3_1R[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 3_1R. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center3_1R_baseline = centers_avg3_1R_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers3_1R_avg = ((centers_avg3_1R_clean - center3_1R_baseline) / center3_1R_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers3_1R_avg,
    mode='lines+markers',
    name='Δ% Center 3_1R',
    line=dict(color='red', width=3) # Manteniamo il rosso per il Peak 3_1R!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 3_1R) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

difference

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Inserisci qui l'elenco degli scan che vuoi includere nel calcolo.
# - Solo lo scan 0: [0]
# - Più scan specifici: [0, 1, 2, 4] (salta il 3)
# - Tutti gli scan fino al 5: list(range(6))
scans_to_include = [0,1,2,3]


# 1. Filtriamo le matrici originali prendendo SOLO gli scan scelti
centers3_1L_sel_avg = np.nanmean(centers3_1L[:, scans_to_include], axis=1)
centers3_1R_sel_avg = np.nanmean(centers3_1R[:, scans_to_include], axis=1)

# 2. Calcoliamo la distanza assoluta tra i due spot (D) per ogni delay time
distanza_centri = np.abs(centers3_1R_sel_avg - centers3_1L_sel_avg)

# 3. CALCOLO DELLA BASELINE
# Facciamo la media di TUTTI i tempi negativi per un riferimento stabile
baseline_indices = np.where(delay_times < -200)[0] # Seleziona i punti pre-pompaggio
if len(baseline_indices) == 0:
    baseline_indices = np.arange(5)

dist_baseline_D0 = np.mean(distanza_centri[baseline_indices])

# 4. Calcoliamo la Variazione Percentuale
# Formula: ((D(t) - D0) / D0) * 100
variazione_perc = ((distanza_centri - dist_baseline_D0) / dist_baseline_D0) * 100


fig_dist = go.Figure()

fig_dist.add_trace(go.Scatter(
    x=delay_times,
    y=variazione_perc,
    mode='lines+markers',
    line=dict(color='green', width=3),
    name=f'Δ Distanza (%) - Scan {scans_to_include}'
))

fig_dist.add_hline(y=0, line_dash="dash", line_color="black")

fig_dist.update_layout(
    title=f"Relative distance 11-12 vs Delay time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Relative distance (%)",
    hovermode="x unified",
    template="plotly_white",
    height=450, width=900
)
fig_dist.show()

## Third spot (2L)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile3['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile3.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile3.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 80
x_max = 115

x_vals = profile3['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile3.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile3.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 80
x_max = 115

x_vals = profile3['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile3.sizes['delay_time']
n_scans = profile3.sizes['scan']

fit_params3_2L = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile3.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params3_2L[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes3_2L = fit_params3_2L[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes3_2L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized Amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params3_2L

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers3_2L = fit_params3_2L[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers3_2L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers3_2L = fit_params3_2L[:, :, 1]

n_delays, n_scans = centers3_2L.shape
mask = np.zeros_like(centers3_2L, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers3_2L, np.nan)
centers_avg3_2L = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers3_2L  '
n_scans = centers3_2L.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg3_2L_clean = np.nanmean(centers3_2L[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 3_2L. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center3_2L_baseline = centers_avg3_2L_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers3_2L_avg = ((centers_avg3_2L_clean - center3_2L_baseline) / center3_2L_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers3_2L_avg,
    mode='lines+markers',
    name='Δ% Center 3_2L',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 3_2L!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 3_2L) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

## Fourth spot (2R)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile3['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile3.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile3.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 435
x_max = 471

x_vals = profile3['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile3.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile3.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 435
x_max = 471

x_vals = profile3['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile3.sizes['delay_time']
n_scans = profile3.sizes['scan']

fit_params3_2R = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile3.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params3_2R[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes3_2R = fit_params3_2R[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes3_2R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params3_2R

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

summed

In [ ]:
import plotly.graph_objects as go
import numpy as np
import warnings

# 1. Inserisci i DUE array di parametri dei due spot simmetrici
parametri_spot_3_C = fit_params3_2L
parametri_spot_3_D = fit_params3_2R

bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values

n_scans = parametri_spot_3_C.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_3_C = parametri_spot_3_C[:, good_scans, 0]
amp_3_D = parametri_spot_3_D[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_3_C = np.nanmean(amp_3_C, axis=1)
mean_pura_3_D = np.nanmean(amp_3_D, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_3_C + mean_pura_3_D

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale

fig = go.Figure()

# singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_3_C / np.nanmean(mean_pura_3_C[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot 3C'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_3_D / np.nanmean(mean_pura_3_D[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot 3D'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (13+14) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers3_2R = fit_params3_2R[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers3_2R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers3_2R = fit_params3_2R[:, :, 1]

n_delays, n_scans = centers3_2R.shape
mask = np.zeros_like(centers3_2R, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers3_2R, np.nan)
centers_avg3_2R = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers3_2R  '
n_scans = centers3_2R.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg3_2R_clean = np.nanmean(centers3_2R[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 3_2R. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center3_2R_baseline = centers_avg3_2R_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers3_2R_avg = ((centers_avg3_2R_clean - center3_2R_baseline) / center3_2R_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers3_2R_avg,
    mode='lines+markers',
    name='Δ% Center 3_2R',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 3_2R!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 3_2R) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Inserisci qui l'elenco degli scan che vuoi includere nel calcolo.
# - Solo lo scan 0: [0]
# - Più scan specifici: [0, 1, 2, 4] (salta il 3)
# - Tutti gli scan fino al 5: list(range(6))
scans_to_include = [0,1,2,3]

# 1. Filtriamo le matrici originali prendendo SOLO gli scan scelti
centers3_2L_sel_avg = np.nanmean(centers3_2L[:, scans_to_include], axis=1)
centers3_2R_sel_avg = np.nanmean(centers3_2R[:, scans_to_include], axis=1)

# 2. Calcoliamo la distanza assoluta tra i due spot (D) per ogni delay time
distanza_centri = np.abs(centers3_2R_sel_avg - centers3_2L_sel_avg)

# 3. CALCOLO DELLA BASELINE
# Facciamo la media di TUTTI i tempi negativi per un riferimento stabile
baseline_indices = np.where(delay_times < -200)[0] # Seleziona i punti pre-pompaggio
if len(baseline_indices) == 0:
    baseline_indices = np.arange(5)

dist_baseline_D0 = np.mean(distanza_centri[baseline_indices])

# 4. Calcoliamo la Variazione Percentuale
# Formula: ((D(t) - D0) / D0) * 100
variazione_perc = ((distanza_centri - dist_baseline_D0) / dist_baseline_D0) * 100


fig_dist = go.Figure()

fig_dist.add_trace(go.Scatter(
    x=delay_times,
    y=variazione_perc,
    mode='lines+markers',
    line=dict(color='green', width=3),
    name=f'Δ Distanza (%) - Scan {scans_to_include}'
))

fig_dist.add_hline(y=0, line_dash="dash", line_color="black")

fig_dist.update_layout(
    title=f"Relative distance 13-14 vs Delay time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Relative distance (%)",
    hovermode="x unified",
    template="plotly_white",
    height=450, width=900
)
fig_dist.show()

# Fourth Line

In [ ]:
import hvplot.xarray
vmax=100
vmin=0

ds_us.im.isel(delay_time=5, scan=5).hvplot.image(x='x', y='y', clim=(vmin, vmax), cmap='viridis', aspect='square', 
                                                 title='To be selected for ROI selection')

Rotate the image

In [ ]:
import numpy as np
import xarray as xr
from scipy.ndimage import rotate

# Le tue nuove coordinate per la quarta linea
x0, y0 = 261, 268
x1, y1 = 154, 406
angle_rad = np.arctan2(y1 - y0, x1 - x0)
rot_angle = np.degrees(angle_rad)
shape_y = ds_us.sizes['y']
shape_x = ds_us.sizes['x']
dummy_img = np.zeros((shape_y, shape_x), dtype=bool)

# Calcoliamo la nuova cornice con il VERO angolo
dummy_rotated = rotate(dummy_img, rot_angle, reshape=True)
new_y, new_x = dummy_rotated.shape

def rotate_2d_safe(arr, angle):
    return rotate(arr, angle, reshape=True, order=1, mode='reflect')

# Apply rotation: usiamo 'dir4' per le dimensioni temporanee
da_rotated_temp_4 = xr.apply_ufunc(     
    rotate_2d_safe,
    ds_us['im'],
    rot_angle, # <-- Ora userà l'angolo giusto!
    input_core_dims=[['y', 'x'], []],   
    output_core_dims=[['y_dir4', 'x_dir4']],
    vectorize=True,
    dask='parallelized',  
    dask_gufunc_kwargs={'output_sizes': {'y_dir4': new_y, 'x_dir4': new_x}},
    output_dtypes=[ds_us['im'].dtype]
)

# 1. Riordiniamo le dimensioni
da_rotated_temp_4 = da_rotated_temp_4.transpose('y_dir4', 'x_dir4', 'delay_time', 'scan')

# 2. Rinominiamo
da_rotated_4 = da_rotated_temp_4.rename({'y_dir4': 'y', 'x_dir4': 'x'})

# 3. Assegniamo le coordinate numeriche
da_rotated_4 = da_rotated_4.assign_coords(
    x=np.arange(new_x),
    y=np.arange(new_y)
)

In [ ]:
vmax = 100
vmin = 0

da_rotated_4.isel(delay_time=0, scan=5).hvplot.image(
    x='x', y='y', 
    cmap='viridis', clim=(vmin, vmax), aspect='square',
    title='To be used for ROI selection'
)

In [ ]:
roi4=da_rotated_4.sel(y=slice(288,314), x=slice(140,540))
profile4 = roi4.mean(dim='y')

## First spot (1L)

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

x_vals = profile4['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile4.sizes['scan']

cols=4
rows = math.ceil(num_scans / cols)

fig= make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile4.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 156
x_max = 205

x_vals = profile4['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile4.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile4.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 156
x_max = 205

x_vals = profile4['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile4.sizes['delay_time']
n_scans = profile4.sizes['scan']

fit_params4_1L = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile4.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params4_1L[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes4_1L = fit_params4_1L[:, j, 0]
    
    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes4_1L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params4_1L

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### Peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers4_1L = fit_params4_1L[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers4_1L,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers4_1L = fit_params4_1L[:, :, 1]

n_delays, n_scans = centers4_1L.shape
mask = np.zeros_like(centers4_1L, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers4_1L, np.nan)
centers_avg4_1L = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers4_1L  '
n_scans = centers4_1L.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg4_1L_clean = np.nanmean(centers4_1L[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 4_1L. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center4_1L_baseline = centers_avg4_1L_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers4_1L_avg = ((centers_avg4_1L_clean - center4_1L_baseline) / center4_1L_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers4_1L_avg,
    mode='lines+markers',
    name='Δ% Center 4_1L',
    line=dict(color='blue', width=3) # Manteniamo il blu per il Peak 4_1L!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 4_1L) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

## Second spot (1R)

In [ ]:
import plotly.graph_objects as go
import numpy as np
import math

x_vals = profile4['x'].values
x_centered = x_vals - x_vals.mean()
num_scans = profile4.sizes['scan']

# Set up grid dimensions
cols = 4
rows = math.ceil(num_scans / cols)

# Create subplot grid
fig = make_subplots(
    rows=rows,
    cols=cols,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.05,
    horizontal_spacing=0.05,
    subplot_titles=[f"Scan {i}" for i in range(num_scans)]
)

# Add traces
for i in range(num_scans):
    row = (i // cols) + 1
    col = (i % cols) + 1
    y_vals = profile4.isel(delay_time=2, scan=i).values
    fig.add_trace(
        go.Scatter(x=x_vals, y=y_vals, mode='lines', line=dict(width=1)),
        row=row,
        col=col
    )

fig.update_layout(
    height=300 * rows,
    width=300 * cols,
    title_text="Intensity vs. x for each scan (Grid View)",
    showlegend=False
)

fig.update_xaxes(title_text="x (pixels)")
fig.update_yaxes(title_text="Mean intensity")

fig.show()

In [ ]:
from scipy.optimize import curve_fit

def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept

x_min = 490
x_max = 535

x_vals = profile4['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 143)  # further limit to avoid second peak
#fit_mask = ((x_vals >= x_min) & (x_vals <= 105)) + ((x_vals >= 115) & (x_vals <= x_max))
x_fit = x_vals[fit_mask]

num_scans = profile4.sizes['scan']
ncols = 4
nrows = int(np.ceil(num_scans / ncols))

fig = make_subplots(rows=nrows, cols=ncols, shared_xaxes=True, shared_yaxes=True,
                    subplot_titles=[f"Scan {i}" for i in range(num_scans)],
                    horizontal_spacing=0.05, vertical_spacing=0.07)

for i in range(num_scans):
    y_vals = profile4.isel(delay_time=10, scan=i).values
    y_fit = y_vals[fit_mask]

    # Initial guess
    A_guess = y_fit.max() - y_fit.min()
    mu_guess = x_fit[np.argmax(y_fit)]
    sigma_guess = 3.0
    slope_guess = 0.0
    intercept_guess = np.mean(y_fit)        # y_fit.min()

    p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

    try:
        popt, _ = curve_fit(gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=2000)
        y_fit_curve = gaussian_with_linear_bg(x_fit, *popt)

        row = (i // ncols) + 1
        col = (i % ncols) + 1

        # Plot raw profile
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit,
                                 mode='lines', name=f"Scan {i} Data",
                                 line=dict(width=1),
                                 showlegend=False),
                      row=row, col=col)

        # Plot fit
        fig.add_trace(go.Scatter(x=x_fit, y=y_fit_curve,
                                 mode='lines', name=f"Scan {i} Fit",
                                 line=dict(dash='dash', width=1),
                                 showlegend=False),
                      row=row, col=col)

    except RuntimeError:
        print(f"Fit failed for scan {i}")

# Layout
fig.update_layout(
    height=250 * nrows,
    width=300 * ncols,
    title="Gaussian + Linear Background Fit (Subset Range) per Scan",
    showlegend=False
)

# fig.update_xaxes(title_text="x (centered)", row=nrows, col=1)
# fig.update_yaxes(title_text="Intensity", row=nrows, col=1)
fig.update_xaxes(title_text="x (centered)")
fig.update_yaxes(title_text="Intensity")

fig.show()

In [ ]:
def gaussian_with_linear_bg(x, A, mu, sigma, slope, intercept):
    return A * np.exp(-(x - mu)**2 / (2 * sigma**2)) + slope * x + intercept
x_min = 490
x_max = 535

x_vals = profile4['x'].values
fit_mask = (x_vals >= x_min) & (x_vals <= x_max)
#fit_mask &= (x_vals <= 100)  # further limit to avoid second peak
x_fit = x_vals[fit_mask]

n_delays = profile4.sizes['delay_time']
n_scans = profile4.sizes['scan']

fit_params4_1R = np.full((n_delays, n_scans, 5), np.nan)        # Array with shape (delay_time, scan, 5 parameters)

for i in range(n_delays):
    for j in range(n_scans):
        y_vals = profile4.isel(delay_time=i, scan=j).values
        y_fit = y_vals[fit_mask]

        # Initial parameter guess
        A_guess =  np.max(y_fit) - np.min(y_fit)
        mu_guess = x_fit[np.argmax(y_fit)]
        sigma_guess = 3.0
        slope_guess = 0.0
        intercept_guess = np.mean(y_fit)
        p0 = [A_guess, mu_guess, sigma_guess, slope_guess, intercept_guess]

        try:
            popt, _ = curve_fit(
                gaussian_with_linear_bg, x_fit, y_fit, p0=p0, maxfev=5000
            )
            fit_params4_1R[i, j, :] = popt  # Store A, mu, sigma, m, b
        except:
            continue  # Skip failed fits

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    amplitudes4_1R = fit_params4_1R[:, j, 0]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=amplitudes4_1R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Amplitude vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Amplitude (A)")
fig.update_xaxes(title_text="Delay Time")

fig.show()

### Normalized Amplitude

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_da_usare = fit_params4_1R

#Quali scan escludere? (Lascia vuoto se vanno tutti bene)
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# 3. Quanti punti usiamo per calcolare lo zero (baseline)?
punti_baseline = 3
# ==========================================

# Estraiamo i ritardi e il numero di scan
delays = profile['delay_time'].values
n_scans = parametri_da_usare.shape[1]

# Estraiamo l'ampiezza pura (A), che si trova all'indice 0 dei parametri
amplitudes = parametri_da_usare[:, :, 0]

# Lista degli scan da tenere
good_scans = [j for j in range(n_scans) if j not in bad_scans]


# 1. Estraiamo solo i dati grezzi degli scan buoni
amplitudes_good = amplitudes[:, good_scans]

# 2. MEDIA PRIMA: Facciamo la media dei dati PURI lungo l'asse degli scan
mean_pura = np.nanmean(amplitudes_good, axis=1)

# 3. NORMALIZZAZIONE DOPO: Calcoliamo la baseline sulla curva già mediata
baseline_media = np.nanmean(mean_pura[:punti_baseline]) 

# 4. Dividiamo la curva media per la sua baseline
mean_normalizzata = mean_pura / baseline_media

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata, 
    mode='lines+markers', 
    line=dict(color='blue', width=3), 
    name=f'Media ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Dinamica del Reticolo: Ampiezza Media Normalizzata",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Ampiezza Normalizzata (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

# Linea di riferimento a 1.0 (baseline pre-pump)
fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

In [ ]:
import plotly.graph_objects as go
import numpy as np
import warnings

# 1. Inserisci i DUE array di parametri dei due spot simmetrici
parametri_spot_4_A = fit_params4_1L
parametri_spot_4_B = fit_params4_1R

bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values

n_scans = parametri_spot_4_A.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_4_A = parametri_spot_4_A[:, good_scans, 0]
amp_4_B = parametri_spot_4_B[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_4_A = np.nanmean(amp_4_A, axis=1)
mean_pura_4_B = np.nanmean(amp_4_B, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_4_A + mean_pura_4_B

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale

fig = go.Figure()

# singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_4_A / np.nanmean(mean_pura_4_A[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot 4A'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_4_B / np.nanmean(mean_pura_4_B[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot 4B'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (15+16) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

### peak center

In [ ]:
n_cols = 4
n_rows = int(np.ceil(n_scans / n_cols))

fig = make_subplots(
    rows=n_rows,
    cols=n_cols,
    subplot_titles=[f"Scan {j}" for j in range(n_scans)],
    horizontal_spacing=0.07,
    vertical_spacing=0.1,
    shared_yaxes=False
)

for j in range(n_scans):
    row = (j // n_cols) + 1
    col = (j % n_cols) + 1

    centers4_1R = fit_params4_1R[:, j, 1]

    fig.add_trace(go.Scatter(
        x=delay_times,
        y=centers4_1R,
        mode='lines+markers',
        name=f"Scan {j}",
        showlegend=False
    ), row=row, col=col)

fig.update_layout(
    height=250 * n_rows,
    width=300 * n_cols,
    title="Peak center vs. Delay Time for Each Scan",
    showlegend=False
)

fig.update_yaxes(title_text="Peak center")
fig.update_xaxes(title_text="Delay Time")

fig.show()

In [ ]:
centers4_1R = fit_params4_1R[:, :, 1]

n_delays, n_scans = centers4_1R.shape
mask = np.zeros_like(centers4_1R, dtype=bool)


included_scans = list(range(n_scans))

# Enable the scans
for j in included_scans:
    mask[:, j] = True

# (Ho lasciato commentata la tua parte per rimuovere punti specifici,
# puoi decommentarla se ti serve!)
# for i, t in enumerate(delay_times):
#     if t == 3 and 4 in included_scans:
#         mask[i, 4] = False

masked_centers = np.where(mask, centers4_1R, np.nan)
centers_avg4_1R = np.nanmean(masked_centers, axis=1)

# --- 1. ESCLUSIONE SCAN E CALCOLO MEDIA PULITA ---
# Inserisci qui dentro le parentesi quadre gli indici degli scan da togliere (es. [4, 5, 9])
# Se la lasci vuota [], fa la media su tutti.
bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 

# Troviamo solo gli indici degli scan "buoni" usando l'array 2D 'centers4_1R  '
n_scans = centers4_1R.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# Ricalcoliamo la media SOLO sulle colonne degli scan buoni
centers_avg4_1R_clean = np.nanmean(centers4_1R[:, good_scans], axis=1)

print(f"✅ Media calcolata su {len(good_scans)} scan per il Peak 4_1R. Esclusi: {bad_scans}")

# --- 2. CALCOLO VARIAZIONE PERCENTUALE ---
# Definiamo la baseline (prendiamo il primo delay, indice 0)
idx_baseline = 0
center4_1R_baseline = centers_avg4_1R_clean[idx_baseline]

# Calcoliamo la variazione percentuale
delta_centers4_1R_avg = ((centers_avg4_1R_clean - center4_1R_baseline) / center4_1R_baseline) * 100

# --- 3. CREIAMO IL GRAFICO ---
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=delay_times,
    y=delta_centers4_1R_avg,
    mode='lines+markers',
    name='Δ% Center 4_1R',
    line=dict(color='red', width=3) # Manteniamo il rosso per il Peak 4_1R!
))

fig.update_layout(
    title=f"Percentage Variation of Average Center (Right Peak 4_1R) vs Delay Time<br><sup>(Excluded scans: {bad_scans})</sup>",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Δ Center / Center₀ (%)",
    hovermode="x unified"
)

fig.show()

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Inserisci qui l'elenco degli scan che vuoi includere nel calcolo.
# - Solo lo scan 0: [0]
# - Più scan specifici: [0, 1, 2, 4] (salta il 3)
# - Tutti gli scan fino al 5: list(range(6))
scans_to_include = [0,1,2,3]


# 1. Filtriamo le matrici originali prendendo SOLO gli scan scelti
centers4_1L_sel_avg = np.nanmean(centers4_1L[:, scans_to_include], axis=1)
centers4_1R_sel_avg = np.nanmean(centers4_1R[:, scans_to_include], axis=1)

# 2. Calcoliamo la distanza assoluta tra i due spot (D) per ogni delay time
distanza_centri = np.abs(centers4_1R_sel_avg - centers4_1L_sel_avg)

# 3. CALCOLO DELLA BASELINE
# Facciamo la media di TUTTI i tempi negativi per un riferimento stabile
baseline_indices = np.where(delay_times < -200)[0] # Seleziona i punti pre-pompaggio
if len(baseline_indices) == 0:
    baseline_indices = np.arange(5)

dist_baseline_D0 = np.mean(distanza_centri[baseline_indices])

# 4. Calcoliamo la Variazione Percentuale
# Formula: ((D(t) - D0) / D0) * 100
variazione_perc = ((distanza_centri - dist_baseline_D0) / dist_baseline_D0) * 100


fig_dist = go.Figure()

fig_dist.add_trace(go.Scatter(
    x=delay_times,
    y=variazione_perc,
    mode='lines+markers',
    line=dict(color='green', width=3),
    name=f'Δ Distanza (%) - Scan {scans_to_include}'
))

fig_dist.add_hline(y=0, line_dash="dash", line_color="black")

fig_dist.update_layout(
    title=f"Relative distance 15-16 vs Delay time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Relative Distance(%)",
    hovermode="x unified",
    template="plotly_white",
    height=450, width=900
)
fig_dist.show()

# Summ of amplitudes equivalent spots

## 1,2,7,8,11,12

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_spot_1 = fit_params  
parametri_spot_2 = fit_params2 
parametri_spot_7= fit_params2_1L
parametri_spot_8= fit_params2_1R
parametri_spot_11= fit_params3_1L
parametri_spot_12= fit_params3_1R

bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values
n_scans = parametri_spot_1.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_1 = parametri_spot_1[:, good_scans, 0]
amp_2 = parametri_spot_2[:, good_scans, 0]
amp_7 = parametri_spot_7[:, good_scans, 0]
amp_8 = parametri_spot_8[:, good_scans, 0]
amp_11 = parametri_spot_11[:, good_scans, 0]
amp_12 = parametri_spot_12[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_1 = np.nanmean(amp_1, axis=1)
mean_pura_2 = np.nanmean(amp_2, axis=1)
mean_pura_7 = np.nanmean(amp_7, axis=1)
mean_pura_8 = np.nanmean(amp_8, axis=1)
mean_pura_11 = np.nanmean(amp_11, axis=1)
mean_pura_12 = np.nanmean(amp_12, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_1 + mean_pura_2 + mean_pura_7 + mean_pura_8 + mean_pura_11 + mean_pura_12

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale


fig = go.Figure()

#singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_1 / np.nanmean(mean_pura_1[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot 1'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_2 / np.nanmean(mean_pura_2[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot 2'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_7 / np.nanmean(mean_pura_7[:punti_baseline]), mode='lines', line=dict(dash='dot', color='black'), name='Spot 7'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_8 / np.nanmean(mean_pura_8[:punti_baseline]), mode='lines', line=dict(dash='dot', color='orange'), name='Spot 8'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_11 / np.nanmean(mean_pura_11[:punti_baseline]), mode='lines', line=dict(dash='dot', color='blue'), name='Spot 11'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_12 / np.nanmean(mean_pura_12[:punti_baseline]), mode='lines', line=dict(dash='dot', color='green'), name='Spot 12'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (1+2+7+8+11+12) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()

## 3,4,9,10,13,14

In [ ]:
import plotly.graph_objects as go
import numpy as np

parametri_spot_3 = fit_params3
parametri_spot_4 = fit_params4 
parametri_spot_9= fit_params2_2L
parametri_spot_10= fit_params2_2R
parametri_spot_13= fit_params3_2L
parametri_spot_14= fit_params3_2R

bad_scans = [4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20] 
punti_baseline = 4
# ==========================================

delays = profile['delay_time'].values
n_scans = parametri_spot_3.shape[1]
good_scans = [j for j in range(n_scans) if j not in bad_scans]

# --- ESTRAZIONE AMPIEZZE (Indice 0) ---
amp_3 = parametri_spot_3[:, good_scans, 0]
amp_4 = parametri_spot_4[:, good_scans, 0]
amp_9 = parametri_spot_9[:, good_scans, 0]
amp_10 = parametri_spot_10[:, good_scans, 0]
amp_13 = parametri_spot_13[:, good_scans, 0]
amp_14 = parametri_spot_14[:, good_scans, 0]

# --- MEDIA LUNGO GLI SCAN ---
mean_pura_3 = np.nanmean(amp_3, axis=1)
mean_pura_4 = np.nanmean(amp_4, axis=1)
mean_pura_9 = np.nanmean(amp_9, axis=1)
mean_pura_10 = np.nanmean(amp_10, axis=1)
mean_pura_13 = np.nanmean(amp_13, axis=1)
mean_pura_14 = np.nanmean(amp_14, axis=1)

# --- SOMMA DELLE INTENSITÀ ---
mean_pura_totale = mean_pura_3 + mean_pura_4 + mean_pura_9 + mean_pura_10 + mean_pura_13 + mean_pura_14

# --- NORMALIZZAZIONE FINALE SULLA SOMMA ---
baseline_totale = np.nanmean(mean_pura_totale[:punti_baseline]) 
mean_normalizzata_totale = mean_pura_totale / baseline_totale


fig = go.Figure()

#singoli spot per confronto
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_3 / np.nanmean(mean_pura_3[:punti_baseline]), mode='lines', line=dict(dash='dot', color='gray'), name='Spot 3'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_4 / np.nanmean(mean_pura_4[:punti_baseline]), mode='lines', line=dict(dash='dot', color='silver'), name='Spot 4'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_9 / np.nanmean(mean_pura_9[:punti_baseline]), mode='lines', line=dict(dash='dot', color='black'), name='Spot 9'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_10 / np.nanmean(mean_pura_10[:punti_baseline]), mode='lines', line=dict(dash='dot', color='orange'), name='Spot 10'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_13 / np.nanmean(mean_pura_13[:punti_baseline]), mode='lines', line=dict(dash='dot', color='blue'), name='Spot 13'))
# fig.add_trace(go.Scatter(x=delays, y=mean_pura_14 / np.nanmean(mean_pura_14[:punti_baseline]), mode='lines', line=dict(dash='dot', color='green'), name='Spot 14'))

fig.add_trace(go.Scatter(
    x=delays, 
    y=mean_normalizzata_totale, 
    mode='lines+markers', 
    line=dict(color='purple', width=3), 
    name=f'Somma Normalizzata ({len(good_scans)} scan)'
))

fig.update_layout(
    title=f"Amplitude Sum (3+4+9+10+13+14) vs Delay Time",
    xaxis_title="Delay Time (ps)",
    yaxis_title="Normalized Amplitude (A / A₀)",
    hovermode="x unified",
    height=500, width=900,
    margin=dict(l=50, r=50, t=50, b=50)
)

fig.add_hline(y=1.0, line_dash="dash", line_color="black", opacity=0.5)

fig.show()